# 00 — Esplorazione dati Track#3
Dataset **imagined speech, 5 classi** (Hello/Helpme/Stop/Thankyou/Yes), 15 soggetti, 64 canali @ 256 Hz, **subject-dependent**. Chance = 20%.

Questo notebook carica i dati grezzi e ne verifica shape, bilanciamento, e mostra segnale/PSD/topomap. Non modifica nulla.

In [ ]:
# --- setup: rende importabili i moduli track3_*.py ---
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_preproc as P
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()

## 1. Carico un soggetto (tutti gli split)

In [ ]:
tr, va, te = io.load_subject_all(1)
for sd in (tr, va, te):
    print(f'{sd.split:6s} X={sd.X.shape} (trials,ch,time)  y={sd.y.shape}  classi={np.bincount(sd.y, minlength=5)}')
print('canali:', sd.clab[:8], '...')
print('fs:', tr.fs, 'Hz | t:', tr.t[[0,-1]], 'ms | pos_3d:', tr.pos_3d.shape)

## 2. Bilanciamento classi su tutti i 15 soggetti
Verifica che ogni soggetto abbia 60/10/10 trial per classe (train/val/test).

In [ ]:
import pandas as pd
rows=[]
for s in C.SUBJECTS:
    tr,va,te = io.load_subject_all(s)
    rows.append({'subj':s,'train':tr.X.shape[0],'val':va.X.shape[0],'test':te.X.shape[0],
                 'train_bal':str(np.bincount(tr.y,minlength=5))})
pd.DataFrame(rows).set_index('subj')

## 3. Esempio di epoca (soggetto 1, una trial per classe)

In [ ]:
sd = tr
fig, axes = plt.subplots(5,1, figsize=(10,9), sharex=True)
for cls in range(5):
    idx = np.where(sd.y==cls)[0][0]
    # media su un sottoinsieme di canali per leggibilità
    axes[cls].plot(sd.t, sd.X[idx, :10, :].T, lw=.6)
    axes[cls].axvline(0, color='k', ls='--', lw=1)
    axes[cls].set_ylabel(C.CLASS_NAMES[cls])
axes[-1].set_xlabel('tempo (ms)  — linea a t=0: onset imagined speech')
fig.suptitle('Esempio epoche grezze (primi 10 canali)')
plt.tight_layout(); plt.show()

## 4. PSD media per classe (Welch)

In [ ]:
from scipy.signal import welch
fig, ax = plt.subplots(figsize=(8,4))
for cls in range(5):
    X = sd.X[sd.y==cls]              # (n, ch, time)
    f, pxx = welch(X, fs=sd.fs, nperseg=256, axis=2)
    ax.semilogy(f, pxx.mean(axis=(0,1)), label=C.CLASS_NAMES[cls])
ax.set_xlim(0,45); ax.set_xlabel('Hz'); ax.set_ylabel('PSD'); ax.legend()
ax.set_title('PSD media per classe — S01'); plt.show()

## 5. Topomap della varianza media (dove è l'energia del segnale)

In [ ]:
import mne
clab, pos = io.canonical_positions()
info = mne.create_info(clab, sd.fs, ch_types='eeg')
# montaggio dalle posizioni 3D fornite (mnt.pos_3d)
montage = mne.channels.make_dig_montage(ch_pos={c:p for c,p in zip(clab,pos)}, coord_frame='head')
info.set_montage(montage, on_missing='warn')
var = sd.X.var(axis=(0,2))           # varianza per canale
fig, ax = plt.subplots(figsize=(4,4))
mne.viz.plot_topomap(var, info, axes=ax, show=False)
ax.set_title('Varianza media per canale — S01'); plt.show()

### Note
- Nessun preprocessing è stato applicato oltre l'epoching (dati grezzi).
- Prossimo step: **01_preprocessing** (filtro, baseline, crop, standardizzazione).